In [2]:
# 1. IMPORT LIBRARIES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configuration
import warnings
warnings.filterwarnings('ignore')

# 2. LOAD DATASET
# Adjust path according to your Kaggle directory structure
train_df = pd.read_csv('https://raw.githubusercontent.com/neerajcodes888/Loan-Approval-Prediction/main/train.csv')
test_df = pd.read_csv('https://raw.githubusercontent.com/neerajcodes888/Loan-Approval-Prediction/main/test.csv')

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape: {test_df.shape}\n")
print("--- First 5 Rows ---")
print(train_df.head())

# 3. DATA CLEANING & MISSING VALUE IMPUTATION
df = train_df.copy()

# Drop Loan_ID as it isn't predictive
df.drop(columns=['Loan_ID'], inplace=True)

# Impute categorical variables with Mode
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Impute numerical variables with Median
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)

# Verify no missing values remain
print("\nRemaining missing values:")
print(df.isnull().sum().sum())

# 4. CATEGORICAL ENCODING & MAPPINGS

# Fixed mappings ensure direct alignment with your Flask HTML form inputs
mappings = {
    'Gender': {'Male': 1, 'Female': 0},
    'Married': {'Yes': 1, 'No': 0},
    'Dependents': {'0': 0, '1': 1, '2': 2, '3+': 3},
    'Education': {'Graduate': 1, 'Not Graduate': 0},
    'Self_Employed': {'Yes': 1, 'No': 0},
    'Property_Area': {'Rural': 0, 'Semiurban': 1, 'Urban': 2},
    'Loan_Status': {'Y': 1, 'N': 0}
}

for col, mapping in mappings.items():
    df[col] = df[col].map(mapping)

# Save mappings for Flask backend verification
joblib.dump(mappings, 'feature_mappings.pkl')

# 5. FEATURE ENGINEERING & TRANSFORMATION
# Combine applicant and co-applicant income
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# Log transformations to reduce right-skewness
df['LoanAmount_Log'] = np.log(df['LoanAmount'])
df['Total_Income_Log'] = np.log(df['Total_Income'])

# Drop original skewed/redundant features
df.drop(columns=['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Total_Income'], inplace=True)

# 6. TRAIN / TEST SPLIT
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

# 7. MODEL TRAINING & EVALUATION
# Initialize models with Bias Correction (Class Weighting)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced')
}

best_model = None
best_accuracy = 0.0

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    
    print(f"\n{"="*10} {name} {"="*10}")
    print(f"Validation Accuracy: {acc * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_val, preds))
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model

# 8. SAVE THE MODEL FOR FLASK
# Save the feature names to ensure Flask passes inputs in the correct order
feature_names = list(X.columns)
joblib.dump(feature_names, 'feature_names.pkl')

# Save the best performing model
joblib.dump(best_model, 'loan_approval_model.pkl')

print("\nSuccessfully saved:")
print("1. 'loan_approval_model.pkl' (Trained Model)")
print("2. 'feature_names.pkl' (Ordered Feature List)")
print("3. 'feature_mappings.pkl' (Input Encoding Dictionary)")

Train Dataset Shape: (614, 13)
Test Dataset Shape: (367, 12)

--- First 5 Rows ---
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urb